# Computational Notebook 08: Valuation Models

## Overview

Valuing cryptocurrencies and blockchain protocols remains one of the most debated topics in finance. Unlike equities with earnings or bonds with coupon payments, crypto assets often lack traditional cash flows, forcing analysts to adapt existing frameworks and invent new ones. This notebook implements seven valuation approaches from first principles: Discounted Cash Flow (DCF) adapted for protocol revenue, the Network Value to Transactions (NVT) ratio, Metcalfe's Law network valuation, the Stock-to-Flow (S2F) scarcity model, the Equation of Exchange (MV=PQ), relative valuation using protocol multiples, and risk-adjusted return metrics.

## Prerequisites
- **Notebook 06**: Market Analysis (price data, returns, technical indicators)
- **Notebook 07**: Mining Economics (hashrate, block rewards, halvings)
- Basic Python programming and familiarity with NumPy

## Learning Objectives

1. Adapt Discounted Cash Flow (DCF) analysis to fee-generating crypto protocols
2. Calculate and interpret the Network Value to Transactions (NVT) ratio for Bitcoin and Ethereum
3. Apply Metcalfe's Law to model network value from active addresses
4. Implement the Stock-to-Flow (S2F) model and understand its assumptions and critiques
5. Use the Equation of Exchange (MV=PQ) to estimate token value from velocity and economic output
6. Perform relative valuation using Total Value Locked (TVL) and revenue multiples across DeFi protocols
7. Calculate risk-adjusted return metrics (Sharpe, Sortino, maximum drawdown) for crypto assets

**Estimated Time:** 4-6 hours

**Related Content:** [Section 02: Bitcoin Deep Dive](../sections/02-bitcoin-deep-dive.md) | [Section 04: Blockchain Economics](../sections/04-blockchain-economics.md)

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True

print("All imports successful!")
print("This notebook implements crypto valuation models using Python and numpy.")

---
## 1. Discounted Cash Flow (DCF) for Crypto Protocols

Traditional DCF values an asset as the present value of its expected future cash flows. For crypto protocols, "cash flows" can be interpreted as:

- **Transaction fees** earned by the protocol (e.g., Ethereum base fees burned via EIP-1559)
- **Protocol revenue** distributed to token holders (e.g., Uniswap fee switch)

> **Definition: Discounted Cash Flow (DCF)** -- A valuation method that estimates the present value of an investment based on its expected future cash flows, discounted at an appropriate rate.

$$PV = \sum_{t=1}^{n} \frac{CF_t}{(1 + r)^t} + \frac{TV}{(1 + r)^n}$$

Where:
- $CF_t$ = Cash flow in period $t$
- $r$ = Discount rate (required rate of return)
- $TV$ = Terminal value
- $n$ = Projection period

**Source:** Damodaran, A. (2012). *Investment Valuation*. Wiley.

In [ ]:
def dcf_valuation(initial_cf: float, growth_rates: List[float],
                  discount_rate: float, terminal_growth: float) -> Dict[str, float]:
    """Perform DCF valuation for a crypto protocol.
    
    Args:
        initial_cf: Current annual cash flow (protocol revenue/fees)
        growth_rates: Year-by-year growth rates for projection period
        discount_rate: Required rate of return (WACC equivalent)
        terminal_growth: Long-term perpetual growth rate
    
    Returns:
        Dict with valuation components
    """
    cash_flows = []
    cf = initial_cf
    
    # Project cash flows
    for g in growth_rates:
        cf *= (1 + g)
        cash_flows.append(cf)
    
    # Discount projected cash flows
    pv_cfs = []
    for t, cf_t in enumerate(cash_flows, 1):
        pv = cf_t / (1 + discount_rate) ** t
        pv_cfs.append(pv)
    
    # Terminal value (Gordon Growth Model)
    terminal_cf = cash_flows[-1] * (1 + terminal_growth)
    terminal_value = terminal_cf / (discount_rate - terminal_growth)
    pv_terminal = terminal_value / (1 + discount_rate) ** len(cash_flows)
    
    total_pv = sum(pv_cfs) + pv_terminal
    
    return {
        "cash_flows": cash_flows,
        "pv_cash_flows": pv_cfs,
        "terminal_value": terminal_value,
        "pv_terminal": pv_terminal,
        "total_pv": total_pv,
        "pv_cf_total": sum(pv_cfs),
        "terminal_pct": pv_terminal / total_pv * 100
    }


# Ethereum DCF: fees burned via EIP-1559
print("=" * 60)
print("DCF VALUATION: ETHEREUM PROTOCOL FEES")
print("=" * 60)

# Ethereum burns ~$2B/year in base fees (synthetic estimate)
eth_initial_fees = 2_000_000_000  # $2B annual fees
eth_growth = [0.40, 0.35, 0.30, 0.25, 0.20, 0.15, 0.10, 0.08, 0.06, 0.05]
eth_discount = 0.15  # 15% discount rate (high risk)
eth_terminal_g = 0.03  # 3% perpetual growth

result = dcf_valuation(eth_initial_fees, eth_growth, eth_discount, eth_terminal_g)

print(f"\nAssumptions:")
print(f"  Current annual fees: ${eth_initial_fees/1e9:.1f}B")
print(f"  Growth trajectory: {[f'{g:.0%}' for g in eth_growth]}")
print(f"  Discount rate: {eth_discount:.0%}")
print(f"  Terminal growth: {eth_terminal_g:.0%}")

print(f"\n{'Year':>6} {'Cash Flow':>14} {'PV of CF':>14}")
print("-" * 38)
for i, (cf, pv) in enumerate(zip(result['cash_flows'], result['pv_cash_flows']), 1):
    print(f"{i:>6} ${cf/1e9:>12.2f}B ${pv/1e9:>12.2f}B")

print(f"\nPV of projected cash flows: ${result['pv_cf_total']/1e9:.2f}B")
print(f"PV of terminal value:       ${result['pv_terminal']/1e9:.2f}B")
print(f"Terminal value as % of total: {result['terminal_pct']:.1f}%")
print(f"\nTotal Protocol Value (DCF):  ${result['total_pv']/1e9:.2f}B")

# Per-token value
eth_supply = 120_000_000  # ~120M ETH
implied_price = result['total_pv'] / eth_supply
print(f"Implied ETH price (120M supply): ${implied_price:,.0f}")

In [ ]:
# Sensitivity analysis: discount rate vs growth
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Sensitivity table
discount_rates = [0.10, 0.12, 0.15, 0.18, 0.20, 0.25]
terminal_growths = [0.01, 0.02, 0.03, 0.04, 0.05]

sensitivity = np.zeros((len(discount_rates), len(terminal_growths)))
for i, dr in enumerate(discount_rates):
    for j, tg in enumerate(terminal_growths):
        r = dcf_valuation(eth_initial_fees, eth_growth, dr, tg)
        sensitivity[i, j] = r['total_pv'] / eth_supply

im = axes[0].imshow(sensitivity, cmap='RdYlGn', aspect='auto')
axes[0].set_xticks(range(len(terminal_growths)))
axes[0].set_xticklabels([f'{tg:.0%}' for tg in terminal_growths])
axes[0].set_yticks(range(len(discount_rates)))
axes[0].set_yticklabels([f'{dr:.0%}' for dr in discount_rates])
axes[0].set_xlabel('Terminal Growth Rate')
axes[0].set_ylabel('Discount Rate')
axes[0].set_title('Implied ETH Price ($)')

for i in range(len(discount_rates)):
    for j in range(len(terminal_growths)):
        axes[0].text(j, i, f'${sensitivity[i,j]:,.0f}',
                     ha='center', va='center', fontsize=8)

# Right: Value breakdown
labels = [f'Year {i+1}' for i in range(10)] + ['Terminal']
values = [pv/1e9 for pv in result['pv_cash_flows']] + [result['pv_terminal']/1e9]
colors = plt.cm.Blues(np.linspace(0.3, 0.9, 10)).tolist() + ['#ff7f0e']

axes[1].bar(labels, values, color=colors)
axes[1].set_ylabel('Present Value ($B)')
axes[1].set_title('DCF Value Breakdown')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('/tmp/dcf_sensitivity.png', dpi=100, bbox_inches='tight')
plt.show()
print("DCF sensitivity analysis complete.")
print("Note: Terminal value dominates -- a common critique of DCF for crypto.")

---
## 2. Network Value to Transactions (NVT) Ratio

The NVT ratio is often called the "P/E ratio of crypto." It compares a network's market capitalization to the value transacted on-chain.

> **Definition: NVT Ratio** -- The ratio of a cryptocurrency's network value (market cap) to its daily transaction volume, expressed in USD. A high NVT suggests overvaluation relative to on-chain utility.

$$NVT = \frac{\text{Network Value (Market Cap)}}{\text{Daily Transaction Volume}}$$

**Interpretation:**
- NVT < 20: Potentially undervalued or high utility
- NVT 20-40: Fair value range
- NVT > 40: Potentially overvalued or speculative premium

**Source:** Kalichkin, D. (2018). "NVT Ratio: A New Way to Value Bitcoin." Cryptolab Capital.

In [ ]:
# Generate synthetic NVT data for Bitcoin and Ethereum
np.random.seed(42)
days = 365 * 3  # 3 years of data

# Bitcoin synthetic data
btc_base_price = 30000
btc_returns = np.random.normal(0.0003, 0.03, days)
btc_prices = btc_base_price * np.cumprod(1 + btc_returns)
btc_supply = 19_500_000
btc_mcap = btc_prices * btc_supply

# Transaction volume correlates with price but with noise
btc_base_vol = 8e9  # $8B daily
btc_vol = btc_base_vol * (btc_prices / btc_base_price) ** 0.5 * np.random.lognormal(0, 0.3, days)
btc_nvt = btc_mcap / btc_vol

# Ethereum synthetic data
eth_base_price = 2000
eth_returns = np.random.normal(0.0004, 0.04, days)
eth_prices = eth_base_price * np.cumprod(1 + eth_returns)
eth_supply_vals = 120_000_000
eth_mcap = eth_prices * eth_supply_vals
eth_base_vol = 5e9
eth_vol = eth_base_vol * (eth_prices / eth_base_price) ** 0.6 * np.random.lognormal(0, 0.35, days)
eth_nvt = eth_mcap / eth_vol

# 90-day smoothed NVT (NVT Signal)
def moving_average(data, window):
    return np.convolve(data, np.ones(window)/window, mode='valid')

btc_nvt_signal = moving_average(btc_nvt, 90)
eth_nvt_signal = moving_average(eth_nvt, 90)

print("=" * 60)
print("NVT RATIO ANALYSIS")
print("=" * 60)

for name, nvt in [("Bitcoin", btc_nvt), ("Ethereum", eth_nvt)]:
    print(f"\n{name}:")
    print(f"  Mean NVT:   {np.mean(nvt):>8.1f}")
    print(f"  Median NVT: {np.median(nvt):>8.1f}")
    print(f"  Std Dev:    {np.std(nvt):>8.1f}")
    print(f"  Current:    {nvt[-1]:>8.1f}")

In [ ]:
# Visualize NVT
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: BTC price and NVT
ax1 = axes[0, 0]
ax1_twin = ax1.twinx()
ax1.plot(btc_prices, 'b-', alpha=0.7, label='BTC Price')
ax1_twin.plot(btc_nvt_signal, 'r-', alpha=0.7, label='NVT Signal (90d)')
ax1_twin.axhline(y=20, color='green', linestyle='--', alpha=0.5)
ax1_twin.axhline(y=40, color='red', linestyle='--', alpha=0.5)
ax1.set_title('Bitcoin: Price vs NVT Signal')
ax1.set_ylabel('Price ($)', color='blue')
ax1_twin.set_ylabel('NVT Signal', color='red')
ax1.legend(loc='upper left')
ax1_twin.legend(loc='upper right')

# Top-right: ETH price and NVT
ax2 = axes[0, 1]
ax2_twin = ax2.twinx()
ax2.plot(eth_prices, 'b-', alpha=0.7, label='ETH Price')
ax2_twin.plot(eth_nvt_signal, 'r-', alpha=0.7, label='NVT Signal (90d)')
ax2_twin.axhline(y=20, color='green', linestyle='--', alpha=0.5)
ax2_twin.axhline(y=40, color='red', linestyle='--', alpha=0.5)
ax2.set_title('Ethereum: Price vs NVT Signal')
ax2.set_ylabel('Price ($)', color='blue')
ax2_twin.set_ylabel('NVT Signal', color='red')
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')

# Bottom-left: NVT distribution
axes[1, 0].hist(btc_nvt, bins=50, alpha=0.6, label='Bitcoin', color='orange')
axes[1, 0].hist(eth_nvt, bins=50, alpha=0.6, label='Ethereum', color='blue')
axes[1, 0].axvline(x=20, color='green', linestyle='--', label='Undervalued threshold')
axes[1, 0].axvline(x=40, color='red', linestyle='--', label='Overvalued threshold')
axes[1, 0].set_xlabel('NVT Ratio')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('NVT Distribution')
axes[1, 0].legend(fontsize=8)

# Bottom-right: NVT vs forward returns
forward_days = 30
btc_forward = (btc_prices[forward_days:] / btc_prices[:-forward_days] - 1) * 100
btc_nvt_trimmed = btc_nvt[:len(btc_forward)]
axes[1, 1].scatter(btc_nvt_trimmed, btc_forward, alpha=0.1, s=5, color='orange')
axes[1, 1].set_xlabel('NVT Ratio')
axes[1, 1].set_ylabel(f'{forward_days}-Day Forward Return (%)')
axes[1, 1].set_title('BTC: NVT vs Forward Returns')
axes[1, 1].axhline(y=0, color='black', linewidth=0.5)
axes[1, 1].set_xlim(0, np.percentile(btc_nvt, 95))

plt.tight_layout()
plt.savefig('/tmp/nvt_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("NVT analysis complete.")

---
## 3. Metcalfe's Law Valuation

Metcalfe's Law states that the value of a network is proportional to the square of its number of users:

$$V \propto n^2$$

Applied to crypto: if active addresses serve as a proxy for users, network value should scale quadratically with active addresses.

> **Definition: Metcalfe's Law** -- The observation that the value of a telecommunications network is proportional to the square of the number of connected users, originally formulated by Robert Metcalfe for Ethernet.

In log-log space this becomes a linear relationship:

$$\log(V) = 2 \cdot \log(n) + c$$

**Source:** Peterson, T. (2018). "Metcalfe's Law as a Model for Bitcoin's Value." *Alternative Investment Analyst Review*.

In [ ]:
# Metcalfe's Law: Fit to Bitcoin active addresses
np.random.seed(123)

# Synthetic historical data (monthly, ~10 years)
months = 120
time = np.arange(months)

# Active addresses grow with adoption cycles
base_addresses = 100_000
address_growth = base_addresses * np.exp(0.03 * time) * (1 + 0.3 * np.sin(time / 12 * np.pi))
active_addresses = address_growth * np.random.lognormal(0, 0.15, months)

# Market cap with Metcalfe relationship + noise
metcalfe_constant = 0.5  # Scaling factor
metcalfe_predicted = metcalfe_constant * active_addresses ** 2 / 1e9  # in billions
actual_mcap = metcalfe_predicted * np.random.lognormal(0, 0.4, months)

# Log-log regression
log_addresses = np.log10(active_addresses)
log_mcap = np.log10(actual_mcap * 1e9)

# Fit: log(V) = slope * log(n) + intercept
slope, intercept = np.polyfit(log_addresses, log_mcap, 1)
fitted_log_mcap = slope * log_addresses + intercept

# R-squared
ss_res = np.sum((log_mcap - fitted_log_mcap) ** 2)
ss_tot = np.sum((log_mcap - np.mean(log_mcap)) ** 2)
r_squared = 1 - ss_res / ss_tot

print("=" * 60)
print("METCALFE'S LAW VALUATION")
print("=" * 60)
print(f"\nLog-log regression: log(V) = {slope:.2f} * log(n) + {intercept:.2f}")
print(f"Metcalfe's Law predicts slope = 2.00")
print(f"Fitted slope: {slope:.2f}")
print(f"R-squared: {r_squared:.4f}")
print(f"\nCurrent active addresses: {active_addresses[-1]:,.0f}")
print(f"Metcalfe predicted market cap: ${10**(slope * np.log10(active_addresses[-1]) + intercept) / 1e9:.1f}B")
print(f"Actual market cap: ${actual_mcap[-1]:.1f}B")

In [ ]:
# Visualize Metcalfe's Law fit
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Log-log scatter with regression
axes[0].scatter(log_addresses, log_mcap, alpha=0.5, s=20, color='blue', label='Observed')
axes[0].plot(log_addresses, fitted_log_mcap, 'r-', linewidth=2,
             label=f'Fit: slope={slope:.2f}, R²={r_squared:.3f}')

# Metcalfe ideal line
ideal_line = 2 * log_addresses + (np.mean(log_mcap) - 2 * np.mean(log_addresses))
axes[0].plot(log_addresses, ideal_line, 'g--', linewidth=1, alpha=0.7,
             label='Metcalfe ideal (slope=2.0)')

axes[0].set_xlabel('log₁₀(Active Addresses)')
axes[0].set_ylabel('log₁₀(Market Cap, $)')
axes[0].set_title("Metcalfe's Law: Log-Log Regression")
axes[0].legend()

# Right: Actual vs Metcalfe predicted over time
axes[1].plot(time, actual_mcap, 'b-', alpha=0.7, label='Actual Market Cap')
axes[1].plot(time, metcalfe_predicted, 'r--', alpha=0.7, label="Metcalfe's Predicted")
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Market Cap ($B)')
axes[1].set_title('Actual vs Metcalfe Predicted Market Cap')
axes[1].legend()
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('/tmp/metcalfe_law.png', dpi=100, bbox_inches='tight')
plt.show()
print("Metcalfe's Law analysis complete.")
print(f"The fitted slope of {slope:.2f} is {'close to' if abs(slope - 2) < 0.3 else 'different from'} the theoretical 2.0.")

---
## 4. Stock-to-Flow (S2F) Model

The Stock-to-Flow model, popularized by pseudonymous analyst PlanB, values Bitcoin based on its scarcity, analogous to precious metals.

> **Definition: Stock-to-Flow (S2F)** -- The ratio of existing supply (stock) to annual production (flow). A higher S2F indicates greater scarcity. Gold has S2F ~62, meaning it would take 62 years of production to double the existing supply.

$$S2F = \frac{\text{Existing Supply}}{\text{Annual Production}}$$

$$\text{Model Price} = e^{a \cdot \ln(S2F) + b}$$

Bitcoin's S2F doubles every ~4 years due to halvings:
- Pre-2012: S2F ~4 (50 BTC/block)
- 2012-2016: S2F ~8 (25 BTC/block)
- 2016-2020: S2F ~27 (12.5 BTC/block)
- 2020-2024: S2F ~56 (6.25 BTC/block)
- 2024-2028: S2F ~120 (3.125 BTC/block)

**Source:** PlanB. (2019). "Modeling Bitcoin Value with Scarcity." *Medium*.

### Critiques of S2F
- Assumes scarcity alone drives value (ignores demand)
- No mechanism for price to follow S2F
- Poor out-of-sample performance post-2021
- Implies Bitcoin will exceed global GDP

In [ ]:
def bitcoin_s2f_model(years: int = 20) -> Dict[str, np.ndarray]:
    """Simulate Bitcoin's Stock-to-Flow over time.
    
    Args:
        years: Number of years from genesis to simulate
    
    Returns:
        Dict with time series of supply, flow, S2F, and model price
    """
    months = years * 12
    supply = np.zeros(months)
    flow = np.zeros(months)
    s2f = np.zeros(months)
    halving_era = np.zeros(months, dtype=int)
    
    blocks_per_month = 4380  # ~144 blocks/day * 30.4 days
    block_reward = 50.0
    halving_interval_months = 48  # Every 4 years
    current_supply = 0.0
    
    for m in range(months):
        era = m // halving_interval_months
        halving_era[m] = era
        current_reward = block_reward / (2 ** era)
        monthly_production = blocks_per_month * current_reward
        current_supply += monthly_production
        annual_production = monthly_production * 12
        
        supply[m] = current_supply
        flow[m] = annual_production
        s2f[m] = current_supply / annual_production if annual_production > 0 else float('inf')
    
    # S2F model: ln(price) = 3.3 * ln(S2F) + 14.6 (fitted parameters from PlanB)
    model_price = np.exp(3.3 * np.log(s2f) + 14.6) / 1e8  # Convert sats to USD
    
    return {
        "months": np.arange(months),
        "supply": supply,
        "flow": flow,
        "s2f": s2f,
        "model_price": model_price,
        "halving_era": halving_era
    }


# Generate S2F data
s2f_data = bitcoin_s2f_model(20)

print("=" * 60)
print("BITCOIN STOCK-TO-FLOW MODEL")
print("=" * 60)

# Show S2F at each halving era
print(f"\n{'Era':>4} {'Block Reward':>14} {'Supply':>14} {'Annual Flow':>14} {'S2F':>8} {'Model Price':>14}")
print("-" * 72)

for era in range(5):
    # Last month of each era
    idx = min((era + 1) * 48 - 1, len(s2f_data['s2f']) - 1)
    reward = 50 / (2 ** era)
    print(f"{era:>4} {reward:>12.4f} BTC {s2f_data['supply'][idx]:>12,.0f} "
          f"{s2f_data['flow'][idx]:>12,.0f} {s2f_data['s2f'][idx]:>7.1f} "
          f"${s2f_data['model_price'][idx]:>12,.0f}")

# Compare with commodities
print(f"\nScarcity Comparison:")
commodities = [("Gold", 62), ("Silver", 22), ("Platinum", 1.1),
               ("Bitcoin (current)", s2f_data['s2f'][-1])]
for name, sf in commodities:
    print(f"  {name:<20} S2F: {sf:>6.1f}")

In [ ]:
# Visualize S2F model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: S2F over time with halving eras
era_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for era in range(5):
    mask = s2f_data['halving_era'] == era
    axes[0].semilogy(s2f_data['months'][mask], s2f_data['s2f'][mask],
                     color=era_colors[era], linewidth=2,
                     label=f'Era {era} ({50/(2**era):.2f} BTC/block)')

# Add commodity references
axes[0].axhline(y=62, color='gold', linestyle='--', alpha=0.7, label='Gold S2F')
axes[0].axhline(y=22, color='silver', linestyle='--', alpha=0.7, label='Silver S2F')
axes[0].set_xlabel('Month from Genesis')
axes[0].set_ylabel('Stock-to-Flow Ratio')
axes[0].set_title('Bitcoin Stock-to-Flow Over Time')
axes[0].legend(fontsize=8)

# Right: S2F vs model price (log-log)
for era in range(5):
    mask = s2f_data['halving_era'] == era
    axes[1].scatter(s2f_data['s2f'][mask], s2f_data['model_price'][mask],
                    color=era_colors[era], alpha=0.5, s=10,
                    label=f'Era {era}')

# Model line
s2f_range = np.logspace(0, 3, 100)
model_line = np.exp(3.3 * np.log(s2f_range) + 14.6) / 1e8
axes[1].plot(s2f_range, model_line, 'k-', linewidth=2, label='S2F Model')

axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('Stock-to-Flow')
axes[1].set_ylabel('Model Price ($)')
axes[1].set_title('S2F Model: Price vs Scarcity')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/stock_to_flow.png', dpi=100, bbox_inches='tight')
plt.show()
print("Stock-to-Flow analysis complete.")
print("Warning: S2F has been widely criticized for poor out-of-sample predictions.")

---
## 5. Equation of Exchange (MV=PQ)

The Equation of Exchange from monetary economics can be applied to value utility tokens:

> **Definition: Equation of Exchange (MV=PQ)** -- An economic identity where M (money supply) times V (velocity of money) equals P (price level) times Q (quantity of goods/services). Applied to crypto: if we know a token's velocity and the economic activity it supports, we can estimate its required market cap.

$$M \cdot V = P \cdot Q$$

Rearranged for token value:
$$M = \frac{PQ}{V}$$

Where:
- $M$ = Required monetary base (network value / market cap)
- $V$ = Token velocity (how many times each token changes hands per period)
- $PQ$ = Total economic value transacted through the network

### The Velocity Problem
High velocity reduces the required monetary base, pushing token price down. This is why many protocols try to reduce velocity through staking, locking, and governance mechanisms.

**Source:** Burniske, C. (2017). "Cryptoasset Valuations." Placeholder VC.

In [ ]:
def equation_of_exchange(pq: float, velocity: float, token_supply: float) -> Dict[str, float]:
    """Value a token using MV=PQ.
    
    Args:
        pq: Total economic value flowing through the network annually ($)
        velocity: Number of times each token changes hands per year
        token_supply: Total token supply
    
    Returns:
        Dict with network value and per-token price
    """
    network_value = pq / velocity  # M = PQ / V
    token_price = network_value / token_supply
    
    return {
        "pq": pq,
        "velocity": velocity,
        "network_value": network_value,
        "token_price": token_price
    }


# Example: Valuing a hypothetical DeFi protocol token
print("=" * 60)
print("EQUATION OF EXCHANGE (MV=PQ) VALUATION")
print("=" * 60)

# Protocol parameters
annual_volume = 50_000_000_000  # $50B annual transaction volume
token_supply_mvpq = 1_000_000_000  # 1B tokens

print(f"\nProtocol: Hypothetical DeFi Token")
print(f"Annual transaction volume (PQ): ${annual_volume/1e9:.0f}B")
print(f"Token supply: {token_supply_mvpq/1e9:.0f}B")

# Velocity sensitivity
print(f"\n{'Velocity':>10} {'Network Value':>16} {'Token Price':>14} {'Note':>30}")
print("-" * 74)

velocity_scenarios = [
    (1, "Maximum hold (unlikely)"),
    (3, "Strong staking incentives"),
    (5, "Moderate velocity"),
    (10, "Active trading"),
    (20, "High velocity (no hold incentive)"),
    (50, "Pure utility (pass-through)"),
]

for v, note in velocity_scenarios:
    result = equation_of_exchange(annual_volume, v, token_supply_mvpq)
    print(f"{v:>10} ${result['network_value']/1e9:>14.1f}B ${result['token_price']:>12.2f}  {note}")

print(f"\nKey insight: A 10x increase in velocity causes a 10x decrease in token value.")
print(f"This is why protocols add staking/locking mechanisms to reduce velocity.")

In [ ]:
# Velocity sensitivity visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Token price vs velocity for different PQ levels
velocities = np.linspace(1, 50, 100)
pq_levels = [10e9, 25e9, 50e9, 100e9, 200e9]

for pq in pq_levels:
    prices = (pq / velocities) / token_supply_mvpq
    axes[0].plot(velocities, prices, linewidth=2, label=f'PQ=${pq/1e9:.0f}B')

axes[0].set_xlabel('Token Velocity')
axes[0].set_ylabel('Implied Token Price ($)')
axes[0].set_title('Token Price vs Velocity (MV=PQ)')
axes[0].legend()
axes[0].set_yscale('log')

# Right: Required PQ for target price at different velocities
target_prices = np.linspace(1, 100, 100)
vel_scenarios = [2, 5, 10, 20]

for v in vel_scenarios:
    required_pq = target_prices * token_supply_mvpq * v
    axes[1].plot(target_prices, required_pq / 1e9, linewidth=2, label=f'V={v}')

axes[1].set_xlabel('Target Token Price ($)')
axes[1].set_ylabel('Required Annual Volume ($B)')
axes[1].set_title('Required PQ for Target Price')
axes[1].legend()

plt.tight_layout()
plt.savefig('/tmp/mvpq_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("Equation of Exchange analysis complete.")

---
## 6. Relative Valuation

Relative valuation compares protocols using standardized multiples, similar to how equities are compared using P/E or EV/Revenue ratios.

> **Definition: Total Value Locked (TVL)** -- The total value of crypto assets deposited in a DeFi protocol's smart contracts. TVL is used as a measure of protocol adoption and trust.

Common crypto multiples:
- **FDV/TVL**: Fully Diluted Valuation to Total Value Locked
- **FDV/Revenue**: Fully Diluted Valuation to annualized protocol revenue
- **FDV/Fees**: Fully Diluted Valuation to total fees generated
- **P/S (Price-to-Sales)**: Market cap to annualized revenue

**Source:** Token Terminal. (2024). "DeFi Protocol Metrics." tokenterminal.com

In [ ]:
# DeFi protocol comparison (synthetic but realistic data)
@dataclass
class ProtocolMetrics:
    """Key metrics for a DeFi protocol."""
    name: str
    category: str
    mcap_b: float       # Market cap in billions
    fdv_b: float        # Fully diluted valuation in billions
    tvl_b: float        # Total Value Locked in billions
    revenue_m: float    # Annual revenue in millions
    fees_m: float       # Annual fees in millions
    users_k: float      # Monthly active users in thousands


protocols = [
    ProtocolMetrics("Uniswap", "DEX", 6.0, 9.5, 5.0, 120, 700, 350),
    ProtocolMetrics("Aave", "Lending", 4.0, 5.5, 12.0, 200, 400, 150),
    ProtocolMetrics("Lido", "Staking", 3.0, 3.5, 15.0, 300, 350, 80),
    ProtocolMetrics("MakerDAO", "Lending", 2.5, 2.8, 8.0, 250, 280, 60),
    ProtocolMetrics("Curve", "DEX", 1.0, 2.0, 3.5, 30, 150, 120),
    ProtocolMetrics("Compound", "Lending", 0.8, 1.2, 2.5, 50, 100, 40),
    ProtocolMetrics("SushiSwap", "DEX", 0.4, 0.6, 0.8, 20, 80, 45),
    ProtocolMetrics("Yearn", "Yield", 0.3, 0.35, 0.5, 40, 60, 15),
]

print("=" * 80)
print("RELATIVE VALUATION: DEFI PROTOCOL MULTIPLES")
print("=" * 80)

print(f"\n{'Protocol':<12} {'Category':<10} {'FDV':>8} {'TVL':>8} {'FDV/TVL':>9} "
      f"{'FDV/Rev':>9} {'FDV/Fees':>10} {'Rev/User':>10}")
print("-" * 80)

fdv_tvl_list = []
fdv_rev_list = []

for p in protocols:
    fdv_tvl = p.fdv_b / p.tvl_b
    fdv_rev = p.fdv_b * 1000 / p.revenue_m  # Convert to same units
    fdv_fees = p.fdv_b * 1000 / p.fees_m
    rev_user = p.revenue_m * 1000 / p.users_k  # Revenue per user
    fdv_tvl_list.append(fdv_tvl)
    fdv_rev_list.append(fdv_rev)
    
    print(f"{p.name:<12} {p.category:<10} ${p.fdv_b:>6.1f}B ${p.tvl_b:>6.1f}B "
          f"{fdv_tvl:>8.2f}x {fdv_rev:>8.1f}x {fdv_fees:>9.1f}x ${rev_user:>8.0f}")

print(f"\n{'Median':>22} {'':>8} {'':>8} {np.median(fdv_tvl_list):>8.2f}x {np.median(fdv_rev_list):>8.1f}x")
print(f"\nProtocols trading below median FDV/TVL may be relatively undervalued.")

In [ ]:
# Visualize relative valuation
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

names = [p.name for p in protocols]
categories = [p.category for p in protocols]
cat_colors = {'DEX': '#1f77b4', 'Lending': '#ff7f0e', 'Staking': '#2ca02c', 'Yield': '#d62728'}
colors = [cat_colors[c] for c in categories]

# FDV/TVL
fdv_tvl = [p.fdv_b / p.tvl_b for p in protocols]
bars = axes[0, 0].barh(names, fdv_tvl, color=colors)
axes[0, 0].axvline(x=np.median(fdv_tvl), color='red', linestyle='--', label='Median')
axes[0, 0].set_xlabel('FDV / TVL')
axes[0, 0].set_title('FDV to TVL Multiple')
axes[0, 0].legend()

# FDV/Revenue
fdv_rev = [p.fdv_b * 1000 / p.revenue_m for p in protocols]
axes[0, 1].barh(names, fdv_rev, color=colors)
axes[0, 1].axvline(x=np.median(fdv_rev), color='red', linestyle='--', label='Median')
axes[0, 1].set_xlabel('FDV / Revenue')
axes[0, 1].set_title('FDV to Revenue Multiple')
axes[0, 1].legend()

# Scatter: TVL vs FDV
for cat in cat_colors:
    mask = [p.category == cat for p in protocols]
    tvl = [p.tvl_b for p, m in zip(protocols, mask) if m]
    fdv = [p.fdv_b for p, m in zip(protocols, mask) if m]
    axes[1, 0].scatter(tvl, fdv, color=cat_colors[cat], s=100, label=cat, zorder=5)

for p in protocols:
    axes[1, 0].annotate(p.name, (p.tvl_b, p.fdv_b), fontsize=8, xytext=(5, 5),
                        textcoords='offset points')

axes[1, 0].set_xlabel('TVL ($B)')
axes[1, 0].set_ylabel('FDV ($B)')
axes[1, 0].set_title('TVL vs FDV')
axes[1, 0].legend()

# Revenue vs Fees
revenues = [p.revenue_m for p in protocols]
fees = [p.fees_m for p in protocols]
rev_pct = [p.revenue_m / p.fees_m * 100 for p in protocols]
axes[1, 1].barh(names, rev_pct, color=colors)
axes[1, 1].set_xlabel('Revenue / Fees (%)')
axes[1, 1].set_title('Protocol Revenue Capture Rate')

plt.tight_layout()
plt.savefig('/tmp/relative_valuation.png', dpi=100, bbox_inches='tight')
plt.show()
print("Relative valuation analysis complete.")

---
## 7. Risk-Adjusted Return Metrics

Raw returns don't account for risk. Risk-adjusted metrics help compare crypto assets with traditional investments on a level playing field.

> **Definition: Sharpe Ratio** -- The average excess return per unit of total volatility. Higher is better. Computed as $(R_p - R_f) / \sigma_p$ where $R_f$ is the risk-free rate.

> **Definition: Sortino Ratio** -- Similar to Sharpe but only penalizes downside volatility: $(R_p - R_f) / \sigma_d$, where $\sigma_d$ is the standard deviation of negative returns only.

> **Definition: Maximum Drawdown (MDD)** -- The largest peak-to-trough decline in portfolio value over a given period. Measures worst-case loss.

$$\text{Sharpe} = \frac{R_p - R_f}{\sigma_p}$$

$$\text{Sortino} = \frac{R_p - R_f}{\sigma_d}$$

$$\text{MDD} = \max_{t} \left( \frac{\text{Peak}_t - \text{Trough}_t}{\text{Peak}_t} \right)$$

In [ ]:
def calculate_risk_metrics(returns: np.ndarray, risk_free_rate: float = 0.05,
                           periods_per_year: int = 365) -> Dict[str, float]:
    """Calculate risk-adjusted return metrics.
    
    Args:
        returns: Array of periodic returns
        risk_free_rate: Annual risk-free rate
        periods_per_year: Number of periods in a year (365 for daily)
    
    Returns:
        Dict with Sharpe, Sortino, max drawdown, and other metrics
    """
    # Annualized return
    total_return = np.prod(1 + returns) - 1
    n_years = len(returns) / periods_per_year
    ann_return = (1 + total_return) ** (1 / n_years) - 1
    
    # Annualized volatility
    ann_vol = np.std(returns) * np.sqrt(periods_per_year)
    
    # Sharpe ratio
    sharpe = (ann_return - risk_free_rate) / ann_vol if ann_vol > 0 else 0
    
    # Sortino ratio (downside deviation only)
    downside_returns = returns[returns < 0]
    downside_vol = np.std(downside_returns) * np.sqrt(periods_per_year) if len(downside_returns) > 0 else 0
    sortino = (ann_return - risk_free_rate) / downside_vol if downside_vol > 0 else 0
    
    # Maximum drawdown
    prices = np.cumprod(1 + returns)
    running_max = np.maximum.accumulate(prices)
    drawdowns = (running_max - prices) / running_max
    max_drawdown = np.max(drawdowns)
    
    # Calmar ratio
    calmar = ann_return / max_drawdown if max_drawdown > 0 else 0
    
    return {
        "total_return": total_return,
        "ann_return": ann_return,
        "ann_volatility": ann_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_drawdown,
        "calmar": calmar,
        "best_day": np.max(returns),
        "worst_day": np.min(returns),
        "pct_positive_days": np.mean(returns > 0) * 100
    }


# Generate synthetic daily returns for different assets (3 years)
np.random.seed(42)
n_days = 365 * 3

assets = {
    "BTC": np.random.normal(0.0005, 0.035, n_days),
    "ETH": np.random.normal(0.0006, 0.045, n_days),
    "SOL": np.random.normal(0.0008, 0.06, n_days),
    "SPY": np.random.normal(0.0004, 0.012, n_days),
    "TLT": np.random.normal(0.0001, 0.008, n_days),
}

print("=" * 90)
print("RISK-ADJUSTED RETURN METRICS")
print("=" * 90)

print(f"\n{'Asset':>6} {'Ann Return':>12} {'Ann Vol':>10} {'Sharpe':>8} {'Sortino':>9} "
      f"{'Max DD':>8} {'Calmar':>8} {'Best Day':>10} {'Worst Day':>11}")
print("-" * 90)

all_metrics = {}
for name, rets in assets.items():
    m = calculate_risk_metrics(rets)
    all_metrics[name] = m
    print(f"{name:>6} {m['ann_return']:>11.1%} {m['ann_volatility']:>9.1%} "
          f"{m['sharpe']:>7.2f} {m['sortino']:>8.2f} {m['max_drawdown']:>7.1%} "
          f"{m['calmar']:>7.2f} {m['best_day']:>9.1%} {m['worst_day']:>10.1%}")

In [ ]:
# Visualize risk-return profiles
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

asset_colors = {'BTC': 'orange', 'ETH': 'blue', 'SOL': 'purple', 'SPY': 'green', 'TLT': 'gray'}

# Top-left: Risk-return scatter
for name, m in all_metrics.items():
    axes[0, 0].scatter(m['ann_volatility'] * 100, m['ann_return'] * 100,
                       s=150, color=asset_colors[name], label=name, zorder=5)
    axes[0, 0].annotate(name, (m['ann_volatility'] * 100, m['ann_return'] * 100),
                        fontsize=10, xytext=(8, 0), textcoords='offset points')

axes[0, 0].set_xlabel('Annualized Volatility (%)')
axes[0, 0].set_ylabel('Annualized Return (%)')
axes[0, 0].set_title('Risk-Return Profile')

# Top-right: Cumulative returns
for name, rets in assets.items():
    cum = np.cumprod(1 + rets)
    axes[0, 1].plot(cum, color=asset_colors[name], label=name, alpha=0.8)

axes[0, 1].set_xlabel('Days')
axes[0, 1].set_ylabel('Cumulative Return')
axes[0, 1].set_title('Cumulative Returns (3 Years)')
axes[0, 1].legend()
axes[0, 1].set_yscale('log')

# Bottom-left: Drawdown
for name, rets in assets.items():
    prices = np.cumprod(1 + rets)
    running_max = np.maximum.accumulate(prices)
    dd = (running_max - prices) / running_max * 100
    axes[1, 0].plot(dd, color=asset_colors[name], label=name, alpha=0.7)

axes[1, 0].set_xlabel('Days')
axes[1, 0].set_ylabel('Drawdown (%)')
axes[1, 0].set_title('Drawdown Over Time')
axes[1, 0].legend()
axes[1, 0].invert_yaxis()

# Bottom-right: Metric comparison bars
metric_names = ['Sharpe', 'Sortino', 'Calmar']
x = np.arange(len(metric_names))
width = 0.15

for i, (name, m) in enumerate(all_metrics.items()):
    vals = [m['sharpe'], m['sortino'], m['calmar']]
    axes[1, 1].bar(x + i * width, vals, width, label=name, color=asset_colors[name])

axes[1, 1].set_xticks(x + width * 2)
axes[1, 1].set_xticklabels(metric_names)
axes[1, 1].set_ylabel('Ratio')
axes[1, 1].set_title('Risk-Adjusted Metrics Comparison')
axes[1, 1].legend()
axes[1, 1].axhline(y=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('/tmp/risk_metrics.png', dpi=100, bbox_inches='tight')
plt.show()
print("Risk-adjusted return analysis complete.")
print("Higher Sharpe/Sortino = better risk-adjusted returns.")
print("Crypto assets show higher returns but also much higher volatility.")

---
## Exercises

### Exercise 1: Multi-Model Valuation Dashboard

Build a function that runs all valuation models (DCF, NVT, Metcalfe, S2F, MV=PQ) on a single asset and presents a consolidated valuation range.

**Hints:**
- Each model produces a different price estimate
- Use the median and interquartile range as a fair value zone
- Visualize the range with a box plot or similar

In [ ]:
class MultiModelValuation:
    """Consolidated valuation using multiple models."""
    
    def __init__(self, asset_name: str) -> None:
        """Initialize with asset name."""
        self.asset_name = asset_name
        self.estimates: Dict[str, float] = {}
    
    def add_dcf_estimate(self, initial_cf: float, growth_rates: List[float],
                         discount_rate: float, terminal_growth: float,
                         token_supply: float) -> None:
        """Add a DCF-based valuation."""
        # YOUR CODE HERE
        pass
    
    def add_nvt_estimate(self, market_cap: float, tx_volume: float,
                         fair_nvt: float, token_supply: float) -> None:
        """Add an NVT-based valuation."""
        # YOUR CODE HERE
        pass
    
    def add_mvpq_estimate(self, pq: float, velocity: float,
                          token_supply: float) -> None:
        """Add an MV=PQ-based valuation."""
        # YOUR CODE HERE
        pass
    
    def get_valuation_range(self) -> Dict[str, float]:
        """Return consolidated valuation statistics."""
        # YOUR CODE HERE
        pass

### Exercise 2: Dynamic NVT Backtester

Build a backtester that generates buy/sell signals based on NVT thresholds and calculates the strategy's performance.

**Hints:**
- Buy when NVT Signal drops below a threshold (undervalued)
- Sell when NVT Signal rises above a threshold (overvalued)
- Compare strategy returns to buy-and-hold

In [ ]:
class NVTBacktester:
    """Backtest trading strategy based on NVT signals."""
    
    def __init__(self, prices: np.ndarray, nvt_signal: np.ndarray,
                 buy_threshold: float = 20, sell_threshold: float = 40) -> None:
        """Initialize backtester."""
        self.prices = prices
        self.nvt_signal = nvt_signal
        self.buy_threshold = buy_threshold
        self.sell_threshold = sell_threshold
    
    def run(self) -> Dict[str, float]:
        """Run the backtest and return performance metrics."""
        # YOUR CODE HERE
        pass
    
    def plot_results(self) -> None:
        """Visualize strategy performance vs buy-and-hold."""
        # YOUR CODE HERE
        pass

### Exercise 3: Portfolio Risk Optimizer

Build a mean-variance optimizer that finds the optimal allocation across crypto and traditional assets to maximize the Sharpe ratio.

**Hints:**
- Use Monte Carlo simulation to generate random portfolios
- Calculate portfolio return and risk for each allocation
- Plot the efficient frontier
- Find the portfolio with maximum Sharpe ratio

In [ ]:
class PortfolioOptimizer:
    """Mean-variance portfolio optimizer for crypto/traditional mix."""
    
    def __init__(self, returns_dict: Dict[str, np.ndarray],
                 risk_free_rate: float = 0.05) -> None:
        """Initialize with asset returns."""
        self.returns_dict = returns_dict
        self.risk_free_rate = risk_free_rate
    
    def simulate_portfolios(self, n_portfolios: int = 10000) -> Dict:
        """Generate random portfolios and calculate risk/return."""
        # YOUR CODE HERE
        pass
    
    def find_optimal(self) -> Dict[str, float]:
        """Find the maximum Sharpe ratio portfolio."""
        # YOUR CODE HERE
        pass
    
    def plot_efficient_frontier(self) -> None:
        """Plot the efficient frontier with optimal portfolio."""
        # YOUR CODE HERE
        pass

---
## Summary

### What You Learned
- [x] Adapting DCF analysis to crypto protocol revenue (fee burns, protocol revenue)
- [x] Calculating and interpreting the NVT ratio as a valuation signal
- [x] Applying Metcalfe's Law to model network value from active users
- [x] Implementing the Stock-to-Flow scarcity model and understanding its limitations
- [x] Using the Equation of Exchange (MV=PQ) to value utility tokens
- [x] Performing relative valuation across DeFi protocols using standardized multiples
- [x] Calculating risk-adjusted return metrics (Sharpe, Sortino, maximum drawdown)

### Key Takeaways
1. **No single model is sufficient** -- crypto assets have characteristics of currencies, commodities, equities, and networks simultaneously
2. **DCF works best for fee-generating protocols** but terminal value dominance is a concern
3. **NVT provides relative value signals** but requires smoothing and context
4. **Metcalfe's Law captures network effects** but the exponent varies and adoption is unpredictable
5. **S2F models scarcity, not demand** -- supply reduction is necessary but not sufficient for price increases
6. **Token velocity is the silent killer** -- high velocity tokens struggle to accrue value
7. **Risk-adjusted metrics matter** -- crypto's high returns come with proportionally high risk

### Further Reading
- Burniske, C. & Tatar, J. (2018). *Cryptoassets: The Innovative Investor's Guide*. McGraw-Hill.
- Damodaran, A. (2022). "The Bitcoin/Crypto Divide." Musings on Markets.
- Willy Woo. (2017). "NVT Ratio -- Detecting Bubbles in Bitcoin." woobull.com

### Next Steps
- [Notebook 09: Privacy & Forensics](09-privacy-forensics.ipynb) -- Chain analysis and privacy techniques
- [Notebook 10: Cryptoeconomic Modeling](10-cryptoeconomic-modeling.ipynb) -- Game theory and mechanism design
- [Section 04: Blockchain Economics](../sections/04-blockchain-economics.md) -- Economic theory behind crypto markets